In [0]:
# Databricks notebook source

from __future__ import annotations
import uuid

dbutils.widgets.text("catalog", "phc_txdot")
catalog = dbutils.widgets.get("catalog").strip()

SCRIPT_NAME = "305_build_gold_dim_item_work_category.py"
RUN_ID = str(uuid.uuid4())

SOURCE_SILVER = f"{catalog}.silver.dim_item_specification"
SOURCE_BRONZE = f"{catalog}.bronze.work_category_raw"
TARGET = f"{catalog}.gold.dim_item_work_category"
LOG = f"{catalog}.staging.pipeline_run_log"

def sql_escape(value: str) -> str:
    return value.replace("'", "''")

def log_run(status, row_count, message):
    row_count_sql = "NULL" if row_count is None else str(row_count)
    spark.sql(f"""
        INSERT INTO {LOG}
        VALUES (
            '{RUN_ID}',
            '{SCRIPT_NAME}',
            '{sql_escape(status)}',
            {row_count_sql},
            '{sql_escape(message)}',
            current_timestamp()
        )
    """)

try:
    spark.sql(f"""
    CREATE OR REPLACE TABLE {TARGET}
    AS
    WITH silver_desc AS (
        SELECT DISTINCT
              specification_code
            , specification_description
            , effective_specification_description
            , effective_specification_description_standardized
        FROM {SOURCE_SILVER}
        WHERE effective_specification_description IS NOT NULL
    )

    , bronze_map AS (
        SELECT
              TRIM(specification_description) AS mapped_specification_description
            , UPPER(TRIM(specification_description)) AS mapped_specification_description_standardized
            , TRIM(item_work_category) AS mapped_item_work_category
        FROM {SOURCE_BRONZE}
        WHERE specification_description IS NOT NULL
          AND item_work_category IS NOT NULL
    )

    , joined AS (
        SELECT
              sd.*
            , bm.mapped_specification_description
            , bm.mapped_item_work_category
        FROM silver_desc sd
        LEFT JOIN bronze_map bm
          ON sd.effective_specification_description_standardized
           = bm.mapped_specification_description_standardized
    )

    , classified AS (
        SELECT
              md5(COALESCE(effective_specification_description_standardized, '')) AS item_work_category_key
            , *

            , CASE
                  WHEN effective_specification_description_standardized IN (
                      'MOBILIZATION',
                      'DURATION SPECIFIC (A+B)',
                      'PREPARING RIGHT OF WAY',
                      'PAYMENT ADJUSTMENT-POS',
                      'SPECIAL DEDUCTION'
                  ) THEN 'PROJECT_LEVEL'

                  WHEN mapped_item_work_category IS NOT NULL THEN mapped_item_work_category

                  WHEN effective_specification_description_standardized LIKE '%MOBILIZATION%' THEN 'PROJECT_LEVEL'
                  WHEN effective_specification_description_standardized LIKE '%PAYMENT ADJUSTMENT%' THEN 'PROJECT_LEVEL'
                  WHEN effective_specification_description_standardized LIKE '%SPECIAL DEDUCTION%' THEN 'PROJECT_LEVEL'
                  WHEN effective_specification_description_standardized LIKE '%ALLOWANCE%' THEN 'PROJECT_LEVEL'
                  WHEN effective_specification_description_standardized LIKE '%FORCE ACCOUNT%' THEN 'PROJECT_LEVEL'
                  WHEN effective_specification_description_standardized LIKE '%DURATION SPECIFIC%' THEN 'PROJECT_LEVEL'
                  WHEN effective_specification_description_standardized LIKE '%A+B%' THEN 'PROJECT_LEVEL'
                  WHEN effective_specification_description_standardized LIKE '%PREPARING RIGHT OF WAY%' THEN 'PROJECT_LEVEL'

                  ELSE 'PROJECT_LEVEL'
              END AS item_work_category

            , CASE
                  WHEN effective_specification_description_standardized IN (
                      'MOBILIZATION',
                      'DURATION SPECIFIC (A+B)',
                      'PREPARING RIGHT OF WAY',
                      'PAYMENT ADJUSTMENT-POS',
                      'SPECIAL DEDUCTION'
                  ) THEN 'OVERRIDE'
                  WHEN mapped_item_work_category IS NOT NULL THEN 'CSV_MAPPING'
                  ELSE 'RULE_DEFAULT'
              END AS item_work_category_source

            , CASE
                  WHEN mapped_item_work_category IS NOT NULL THEN FALSE
                  ELSE TRUE
              END AS is_defaulted_work_category

        FROM joined
    )

    SELECT
          item_work_category_key
        , specification_code
        , specification_description
        , effective_specification_description
        , effective_specification_description_standardized
        , item_work_category
        , item_work_category_source
        , is_defaulted_work_category
        , mapped_specification_description
        , mapped_item_work_category
    FROM classified
    """)

    spark.sql(f"""
    COMMENT ON TABLE {TARGET} IS
    'Gold item work category dimension built from silver.dim_item_specification and bronze.work_category_raw.'
    """)

    row_count = spark.sql(f"SELECT COUNT(*) AS c FROM {TARGET}").collect()[0]["c"]
    log_run("SUCCESS", row_count, "Built gold.dim_item_work_category successfully.")

    print("=" * 70)
    print("Built gold.dim_item_work_category")
    print(f"Row count: {row_count:,}")
    print("=" * 70)

except Exception as e:
    log_run("FAILED", None, str(e))
    raise